In [1]:
import os
for r, d, f in os.walk("/kaggle/input"):
    if len(f) > 0:
        print(r, "| files:", len(f))

/kaggle/input/datasets/lilyblossom3/datathon-mock-test/test-20261004T163014Z-1-001/test/all_test | files: 1070
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Macular Scar | files: 355
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Pterygium | files: 13
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Disc Edema | files: 101
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Central Serous Chorioretinopathy | files: 80
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Glaucoma | files: 1111
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Healthy | files: 850
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Retinal Detachment | files: 100
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Retinitis Pigmentosa | files: 111
/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train/Myopia | files: 400
/kaggle/input/datasets/lilyblossom3/dat

In [2]:
import os, pandas as pd, torch

TRAIN_DIR = "/kaggle/input/datasets/lilyblossom3/datathon-mock-test/train/train"
TEST_DIR  = "/kaggle/input/datasets/lilyblossom3/datathon-mock-test/test-20261004T163014Z-1-001/test/all_test"

class_map = {
 "Central Serous Chorioretinopathy": 0,
 "Diabetic Retinopathy": 1,
 "Disc Edema": 2,
 "Glaucoma": 3,
 "Healthy": 4,
 "Macular Scar": 5,
 "Myopia": 6,
 "Pterygium": 7,
 "Retinal Detachment": 8,
 "Retinitis Pigmentosa": 9,
}

rows = []
for cname, label in class_map.items():
    folder = os.path.join(TRAIN_DIR, cname)
    for f in os.listdir(folder):
        rows.append({"path": os.path.join(folder, f), "label": label, "class": cname})

df = pd.DataFrame(rows)
print("train shape:", df.shape)
print(df["label"].value_counts().sort_index())

test_files = os.listdir(TEST_DIR)
print("test images:", len(test_files))
print("first test names:", sorted(test_files)[:10])
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

train shape: (4328, 3)
label
0      80
1    1207
2     101
3    1111
4     850
5     355
6     400
7      13
8     100
9     111
Name: count, dtype: int64
test images: 1070
first test names: ['1.jpg', '10.jpg', '100.jpg', '1000.jpg', '1001.jpg', '1002.jpg', '1003.jpg', '1004.jpg', '1005.jpg', '1006.jpg']
GPU: True Tesla T4


In [3]:
import numpy as np
from PIL import Image

ids = sorted(int(os.path.splitext(f)[0]) for f in test_files)
print("test id min/max:", ids[0], ids[-1], "| count:", len(ids), "| unique:", len(set(ids)))

sizes = [Image.open(p).size for p in df.sample(200, random_state=0).path]
print("sample train image sizes:", sorted(set(sizes))[:10], "... unique:", len(set(sizes)))

test id min/max: 1 1070 | count: 1070 | unique: 1070
sample train image sizes: [(2004, 1690)] ... unique: 1


In [4]:
for cname in class_map:
    names = sorted(os.listdir(os.path.join(TRAIN_DIR, cname)))
    print(cname, "->", names[:6])

Central Serous Chorioretinopathy -> ['CSCR1.jpg', 'CSCR100.jpg', 'CSCR101.jpg', 'CSCR11.jpg', 'CSCR12.jpg', 'CSCR13.jpg']
Diabetic Retinopathy -> ['DR1.jpg', 'DR10.jpg', 'DR1001.jpg', 'DR1002.jpg', 'DR1004.jpg', 'DR1005.jpg']
Disc Edema -> ['Disc Edema1.jpg', 'Disc Edema10.jpg', 'Disc Edema100.jpg', 'Disc Edema101.jpg', 'Disc Edema103.jpg', 'Disc Edema104.jpg']
Glaucoma -> ['Glaucoma1.jpg', 'Glaucoma10.jpg', 'Glaucoma100.jpg', 'Glaucoma1001.jpg', 'Glaucoma1002.jpg', 'Glaucoma1004.jpg']
Healthy -> ['Healthy1.jpg', 'Healthy10.jpg', 'Healthy100.jpg', 'Healthy1000.jpg', 'Healthy1001.jpg', 'Healthy1002.jpg']
Macular Scar -> ['Macular Scar10.jpg', 'Macular Scar101.jpg', 'Macular Scar103.jpg', 'Macular Scar105.jpg', 'Macular Scar106.jpg', 'Macular Scar107.jpg']
Myopia -> ['Myopia10.jpg', 'Myopia100.jpg', 'Myopia101.jpg', 'Myopia102.jpg', 'Myopia104.jpg', 'Myopia105.jpg']
Pterygium -> ['Pterygium1.jpg', 'Pterygium10.jpg', 'Pterygium12.jpg', 'Pterygium14.jpg', 'Pterygium15.jpg', 'Pterygium16.jp

In [5]:
from concurrent.futures import ThreadPoolExecutor

SIZE = 384
os.makedirs("/kaggle/working/train_small", exist_ok=True)
os.makedirs("/kaggle/working/test_small", exist_ok=True)

def shrink(args):
    src, dst = args
    Image.open(src).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR).save(dst, quality=95)

# train
df["small"] = [f"/kaggle/working/train_small/{i}.jpg" for i in range(len(df))]
with ThreadPoolExecutor(8) as ex:
    list(ex.map(shrink, zip(df.path, df.small)))

# test (keep the numeric id in the file name)
test_ids = sorted(int(os.path.splitext(f)[0]) for f in test_files)
with ThreadPoolExecutor(8) as ex:
    list(ex.map(shrink, [(f"{TEST_DIR}/{i}.jpg", f"/kaggle/working/test_small/{i}.jpg") for i in test_ids]))

print("done:", len(os.listdir("/kaggle/working/train_small")), len(os.listdir("/kaggle/working/test_small")))

done: 4328 1070


In [6]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

def thumb(p):
    return np.asarray(Image.open(p).convert("L").resize((16, 16), Image.BILINEAR), dtype=np.float32)

def norm(X):
    X = X.reshape(len(X), -1)
    X = X - X.mean(1, keepdims=True)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-6)

def variants(X):  # 4 rotations x flip = 8 versions
    out = []
    for k in range(4):
        r = np.rot90(X, k, axes=(1, 2))
        out += [r, r[:, :, ::-1]]
    return out

tr = np.stack([thumb(p) for p in df.small])
te = np.stack([thumb(f"/kaggle/working/test_small/{i}.jpg") for i in test_ids])

A = norm(tr)
S = np.max([A @ norm(v).T for v in variants(tr)], axis=0)
np.fill_diagonal(S, 0)

THRESH = 0.98
n, groups = connected_components(csr_matrix(S > THRESH), directed=False)
df["group"] = groups
print("groups:", n, "of", len(df), "images")
print(df.groupby("group").size().value_counts().sort_index().head(10))

St = np.max([norm(te) @ norm(v).T for v in variants(tr)], axis=0)
print("test images with a near-duplicate in train:", int((St.max(1) > THRESH).sum()), "of", len(te))

groups: 401 of 4328 images
1       367
2        27
3         4
4         1
1554      1
2337      1
Name: count, dtype: int64
test images with a near-duplicate in train: 971 of 1070


In [7]:
nn_tr = S.max(1)
nn_te = St.max(1)
print("train nearest-neighbour similarity percentiles (10/50/90/99):", np.percentile(nn_tr, [10, 50, 90, 99]).round(4))
print("test  nearest-neighbour similarity percentiles (10/50/90/99):", np.percentile(nn_te, [10, 50, 90, 99]).round(4))

for t in [0.98, 0.99, 0.995, 0.998, 0.999]:
    n, g = connected_components(csr_matrix(S > t), directed=False)
    sizes = np.bincount(g)
    print(f"thr {t}: groups={n}, largest group={sizes.max()}, test imgs with dup in train={(nn_te > t).sum()}")

train nearest-neighbour similarity percentiles (10/50/90/99): [0.9826 0.9959 1.     1.    ]
test  nearest-neighbour similarity percentiles (10/50/90/99): [0.982  0.9958 1.     1.    ]
thr 0.98: groups=401, largest group=2337, test imgs with dup in train=971
thr 0.99: groups=977, largest group=1747, test imgs with dup in train=839
thr 0.995: groups=2149, largest group=1205, test imgs with dup in train=599
thr 0.998: groups=3496, largest group=141, test imgs with dup in train=289
thr 0.999: groups=3869, largest group=4, test imgs with dup in train=195


In [8]:
n, g = connected_components(csr_matrix(S > 0.999), directed=False)
df["group"] = g

from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df["fold"] = -1
for f, (tri, vai) in enumerate(sgkf.split(df, df.label, df.group)):
    df.loc[df.index[vai], "fold"] = f

print(pd.crosstab(df.fold, df.label))
df.to_csv("/kaggle/working/df.csv", index=False)   # backup in case the session restarts

label   0    1   2    3    4   5   6  7   8   9
fold                                           
0      18  244  18  226  167  70  76  2  20  21
1      15  240  22  216  175  70  84  3  21  25
2      16  241  20  228  174  70  88  2  19  18
3      12  240  18  225  170  74  77  2  21  24
4      19  242  23  216  164  71  75  4  19  23


In [9]:
import numpy as np, pandas as pd, torch, timm, torch.nn as nn, os
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import f1_score, classification_report

torch.cuda.empty_cache()
if "df" not in globals():
    df = pd.read_csv("/kaggle/working/df.csv")
test_ids = sorted(int(os.path.splitext(f)[0]) for f in os.listdir("/kaggle/working/test_small"))

S_, CROP, EPOCHS, FOLD, FULL, dev = 256, 224, 40, 0, False, "cuda"

def load(p):
    return np.asarray(Image.open(p).convert("RGB").resize((S_, S_)), dtype=np.uint8)

with ThreadPoolExecutor(8) as ex:
    Xall = np.stack(list(ex.map(load, df.small)))
    Xte = np.stack(list(ex.map(load, [f"/kaggle/working/test_small/{i}.jpg" for i in test_ids])))

yall = df.label.values
tr_idx = np.arange(len(df)) if FULL else np.where(df.fold != FOLD)[0]
va_idx = np.where(df.fold == FOLD)[0]

def tg(X):
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous().to(dev)

Xtr, ytr = tg(Xall[tr_idx]), torch.tensor(yall[tr_idx]).to(dev)
Xva, yva = tg(Xall[va_idx]), yall[va_idx]
Xt = tg(Xte)

mean = torch.full((1, 3, 1, 1), 127.5, device=dev)
std = torch.full((1, 3, 1, 1), 63.75, device=dev)

def prep(x):
    return (x.float() - mean) / std

def aug(x):
    B = x.shape[0]
    i, j = np.random.randint(0, S_ - CROP + 1, 2)
    x = prep(x[:, :, i:i+CROP, j:j+CROP])
    m = torch.rand(B, device=dev) < 0.5
    x = torch.where(m.view(-1, 1, 1, 1), x.flip(3), x)
    m = torch.rand(B, device=dev) < 0.5
    x = torch.where(m.view(-1, 1, 1, 1), x.flip(2), x)
    x = torch.rot90(x, np.random.randint(4), (2, 3))
    scale = 1 + 0.4 * (torch.rand(B, 1, 1, 1, device=dev) - 0.5)
    shift = 0.4 * (torch.rand(B, 1, 1, 1, device=dev) - 0.5)
    return x * scale + shift

def center(x):
    o = (S_ - CROP) // 2
    return prep(x[:, :, o:o+CROP, o:o+CROP])

@torch.no_grad()
def predict(X, tta=True):
    model.eval()
    out = []
    for k in range(0, len(X), 128):
        x = center(X[k:k+128])
        if tta:
            views = [x, x.flip(3), x.flip(2), torch.rot90(x, 1, (2, 3))]
        else:
            views = [x]
        p = 0
        for v in views:
            with torch.autocast("cuda", dtype=torch.float16):
                p = p + model(v).float().softmax(1)
        out.append(p / len(views))
    return torch.cat(out).cpu().numpy()

model = timm.create_model("resnet18", pretrained=False, num_classes=10, drop_rate=0.2)
model = model.to(dev).to(memory_format=torch.channels_last)
print("ready:", Xtr.shape, Xva.shape, Xt.shape)

ready: torch.Size([3466, 3, 256, 256]) torch.Size([862, 3, 256, 256]) torch.Size([1070, 3, 256, 256])


In [10]:
cnt = np.bincount(ytr.cpu().numpy(), minlength=10)
w = torch.tensor((cnt.max() / cnt) ** 0.5, dtype=torch.float, device=dev)
crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
BS = 64
steps = len(tr_idx) // BS
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS * steps)
scaler = torch.cuda.amp.GradScaler()

for ep in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(steps):
        b = perm[s*BS:(s+1)*BS]
        x = aug(Xtr[b]).contiguous(memory_format=torch.channels_last)
        opt.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = crit(model(x), ytr[b])
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
    if ep % 5 == 4 or ep == EPOCHS - 1:
        P = predict(Xva, False).argmax(1)
        f1 = f1_score(yva, P, average="macro")
        print("epoch", ep + 1, "| loss", round(loss.item(), 3), "| val macro-F1", round(f1, 4), "| acc", round((P == yva).mean(), 4))

P = predict(Xva, True).argmax(1)
print("TTA val macro-F1:", round(f1_score(yva, P, average="macro"), 4))
print(classification_report(yva, P, digits=3))

pt = predict(Xt, True).argmax(1)
sub = pd.DataFrame({"id": test_ids, "target": pt.astype(int)})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.shape, sub.target.value_counts().sort_index().to_dict())

/tmp/ipykernel_23/3737462723.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 5 | loss 1.759 | val macro-F1 0.2082 | acc 0.3063
epoch 10 | loss 1.889 | val macro-F1 0.1789 | acc 0.2343
epoch 15 | loss 1.695 | val macro-F1 0.3348 | acc 0.4942
epoch 20 | loss 1.348 | val macro-F1 0.4891 | acc 0.5534
epoch 25 | loss 1.17 | val macro-F1 0.5006 | acc 0.5824
epoch 30 | loss 1.121 | val macro-F1 0.6588 | acc 0.739
epoch 35 | loss 1.202 | val macro-F1 0.6976 | acc 0.7436
epoch 40 | loss 1.377 | val macro-F1 0.7086 | acc 0.7413
TTA val macro-F1: 0.7004
              precision    recall  f1-score   support

           0      0.250     0.222     0.235        18
           1      0.927     0.832     0.877       244
           2      0.577     0.833     0.682        18
           3      0.740     0.730     0.735       226
           4      0.694     0.802     0.744       167
           5      0.604     0.414     0.492        70
           6      0.614     0.671     0.642        76
           7      1.000     1.000     1.000         2
           8      0.633     0.950  

In [11]:
import time
FOLDS, EPOCHS = [0, 1, 2, 3, 4], 60
BS = 64
test_prob = 0
fold_f1 = []
t0 = time.time()

for FOLD in FOLDS:
    tr_i = np.where(df.fold != FOLD)[0]
    va_i = np.where(df.fold == FOLD)[0]
    Xtr, ytr = tg(Xall[tr_i]), torch.tensor(yall[tr_i]).to(dev)
    Xva, yva = tg(Xall[va_i]), yall[va_i]

    model = timm.create_model("resnet18", pretrained=False, num_classes=10, drop_rate=0.2)
    model = model.to(dev).to(memory_format=torch.channels_last)

    cnt = np.bincount(ytr.cpu().numpy(), minlength=10)
    w = torch.tensor((cnt.max() / cnt) ** 0.5, dtype=torch.float, device=dev)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    steps = len(tr_i) // BS
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS * steps)
    scaler = torch.amp.GradScaler("cuda")

    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(Xtr), device=dev)
        for s in range(steps):
            b = perm[s*BS:(s+1)*BS]
            x = aug(Xtr[b]).contiguous(memory_format=torch.channels_last)
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(model(x), ytr[b])
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

    f1 = f1_score(yva, predict(Xva, True).argmax(1), average="macro")
    fold_f1.append(f1)
    test_prob = test_prob + predict(Xt, True)
    print("fold", FOLD, "| macro-F1", round(f1, 4), "| minutes so far", round((time.time() - t0) / 60, 1))
    del Xtr, Xva
    torch.cuda.empty_cache()

print("mean fold macro-F1:", round(float(np.mean(fold_f1)), 4))
pt = test_prob.argmax(1)
sub = pd.DataFrame({"id": test_ids, "target": pt.astype(int)})
sub.to_csv("/kaggle/working/submission_ens.csv", index=False)
print(sub.shape, sub.target.value_counts().sort_index().to_dict())

fold 0 | macro-F1 0.7488 | minutes so far 3.9
fold 1 | macro-F1 0.7293 | minutes so far 7.8
fold 2 | macro-F1 0.7349 | minutes so far 11.7
fold 3 | macro-F1 0.7272 | minutes so far 15.6
fold 4 | macro-F1 0.765 | minutes so far 19.5
mean fold macro-F1: 0.741
(1070, 2) {0: 17, 1: 295, 2: 31, 3: 231, 4: 240, 5: 79, 6: 116, 7: 4, 8: 26, 9: 31}
